# Notebook 1: N-grams and the Sparsity Problem

> **Article section:** *Part 1 — The Statistical Era*

In this notebook we'll build an N-gram model completely from scratch — no libraries, just Python and math.  
By the end, you'll see *exactly* why N-grams hit a wall, and why Laplace smoothing is a patch rather than a real fix.

---
**What we cover:**
1. Tokenize a corpus
2. Build unigram, bigram, and trigram probability tables
3. Visualise the sparsity problem
4. Add Laplace smoothing and compare
5. Generate text and see its limitations
---

In [ ]:
# Standard library imports — no pip install needed for this cell
import re
import math
import sys
import os
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Allow imports from src/
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.ngram_model import NGramModel

print('All imports OK.')

## 1. Load and tokenise the corpus

In [ ]:
# Load our toy corpus
corpus_path = os.path.join('..', 'data', 'sample_corpus.txt')

with open(corpus_path, 'r') as f:
    raw_lines = f.readlines()

def tokenize(text: str):
    """Lowercase and split on non-alphabetic characters."""
    text = text.lower().strip()
    tokens = re.findall(r"[a-z']+", text)
    return tokens

sentences = [tokenize(line) for line in raw_lines if line.strip()]

print(f'Loaded {len(sentences)} sentences.')
print(f'Example: {sentences[0]}')

## 2. Build probability tables manually

Before using our clean `NGramModel` class, let's build the tables by hand so you can see exactly what's happening.

In [ ]:
# ---- UNIGRAM model ----
all_tokens = [token for sent in sentences for token in sent]
unigram_counts = Counter(all_tokens)
total_tokens = sum(unigram_counts.values())

print(f'Total tokens: {total_tokens}')
print(f'Vocabulary size: {len(unigram_counts)}')
print()
print('Top 10 most common words (unigrams):')
for word, count in unigram_counts.most_common(10):
    prob = count / total_tokens
    print(f'  {word:15s}  count={count:3d}  P={prob:.4f}')

In [ ]:
# ---- BIGRAM model ----
bigram_counts = defaultdict(Counter)

for sent in sentences:
    padded = ['<s>'] + sent + ['</s>']
    for i in range(len(padded) - 1):
        context = padded[i]
        next_word = padded[i + 1]
        bigram_counts[context][next_word] += 1

def bigram_probability(context, word, smoothing=0.0):
    vocab_size = len(unigram_counts) + 2  # +2 for <s>, </s>
    count_cw = bigram_counts[context].get(word, 0)
    count_c  = sum(bigram_counts[context].values())
    return (count_cw + smoothing) / (count_c + smoothing * vocab_size + 1e-9)

# Demonstrate
test_word = 'bank'
print(f"P(bank | 'the')  = {bigram_probability('the', 'bank'):.4f}")
print(f"P(river | 'the') = {bigram_probability('the', 'river'):.4f}")
print(f"P(cat | 'the')   = {bigram_probability('the', 'cat'):.4f}")

## 3. Visualise the Sparsity Problem

Let's plot the bigram probability table as a heatmap.  
Notice how most cells are **zero** — that's the sparsity problem.

In [ ]:
# Use the 12 most common words to build a visible matrix
top_words = [w for w, _ in unigram_counts.most_common(12)]

matrix = np.zeros((len(top_words), len(top_words)))
for i, context in enumerate(top_words):
    for j, word in enumerate(top_words):
        matrix[i, j] = bigram_probability(context, word)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, title, smooth in zip(
    axes,
    ['Raw MLE (no smoothing) — lots of zeros', 'After Laplace Smoothing — zeros filled in'],
    [0.0, 1.0]
):
    m = np.zeros((len(top_words), len(top_words)))
    for i, ctx in enumerate(top_words):
        for j, w in enumerate(top_words):
            m[i, j] = bigram_probability(ctx, w, smoothing=smooth)

    sns.heatmap(
        m,
        ax=ax,
        xticklabels=top_words,
        yticklabels=top_words,
        cmap='Blues',
        annot=True,
        fmt='.2f',
        linewidths=0.3,
        cbar=False,
    )
    ax.set_title(title, fontsize=11, pad=10)
    ax.set_xlabel('Next word', fontsize=9)
    ax.set_ylabel('Context word', fontsize=9)
    ax.tick_params(labelsize=8)

plt.suptitle(
    'Bigram Probability Table: Sparsity vs Laplace Smoothing',
    fontsize=13, fontweight='medium', y=1.02
)
plt.tight_layout()
plt.savefig('../outputs/01_sparsity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to outputs/01_sparsity_heatmap.png')

## 4. N-gram Language Model class

Now let's use the clean `NGramModel` from `src/` to train bigram and trigram models.

In [ ]:
# Train models with and without smoothing
bigram_mle    = NGramModel(n=2, smoothing=0.0).fit(sentences)
bigram_smooth = NGramModel(n=2, smoothing=1.0).fit(sentences)
trigram_mle   = NGramModel(n=3, smoothing=0.0).fit(sentences)

print('Models fitted:', bigram_mle, bigram_smooth, trigram_mle, sep='\n  ')

In [ ]:
# What comes after 'the'?
print('Top words after "the" (bigram, MLE):')
for word, prob in bigram_mle.get_top_next_words(('the',), top_k=8):
    print(f'  {word:15s}  P={prob:.4f}')

In [ ]:
# Demonstrate that 'bank' has NO semantic understanding in N-grams
# The model gives the same next-word distribution regardless of meaning

print('Top words after "bank" (bigram, MLE):')
print('  (Notice: no distinction between river bank vs financial bank)')
for word, prob in bigram_mle.get_top_next_words(('bank',), top_k=8):
    print(f'  {word:15s}  P={prob:.4f}')

## 5. Generate text

Let's generate some sentences. You'll notice the N-gram model produces grammatically reasonable but semantically hollow text.

In [ ]:
import random
random.seed(7)

print('Generated sentences from bigram model:')
for _ in range(5):
    print(' >', bigram_mle.generate(max_words=12))

print()
print('Generated sentences from trigram model:')
for _ in range(5):
    print(' >', trigram_mle.generate(max_words=12))

## 6. Key takeaways

| Problem | Description | Fixed by N-gram? |
|---------|-------------|------------------|
| **Sparsity** | Unseen sequences get zero probability | Partially — Laplace helps |
| **No semantic understanding** | "bank" (river) = "bank" (money) | ❌ No |
| **Short memory** | Can't track dependencies > n words back | ❌ No |
| **No world knowledge** | Doesn't know a fox is an animal | ❌ No |

**Next:** We'll see how Word2Vec gives words *meaning* by representing them as vectors.